# 2 · Dataset construction

From the two panels to the two datasets the models read. This is where the
**target** is defined, and where the look-ahead bias the paper measures is either
kept out or let in on purpose.

Each step is one call to a function of `src/preprocessing.py`, with what it does
above it. The command line does the same in one go:

```bash
python scripts/pipeline.py datasets
python scripts/pipeline.py datasets --example            # on the synthetic panels
python scripts/pipeline.py datasets --missing-threshold 0.4
```

## The two datasets

|  | `window` | `nowindow` |
|---|---|---|
| panel | timed | snapshot |
| attributes | of the row's own year | as declared at extraction time |
| features | read at the age the firm first reached an early stage | cumulated over its whole observed life |
| target | reaches the next stage **within 7 years** of that age | **ever** reaches the next stage |

Everything that makes the second one biased is in it at once, which is why the two
controls further down take that sum apart.

In [1]:
import polars as pl
import yaml

from src.preprocessing import (
    FEATURE_COLUMNS,
    TARGET_COLUMNS,
    build_full_history_dataset,
    build_processed_datasets,
    build_windowed_dataset,
    preprocess_dataset,
)

# EXAMPLE reads the panels built from the synthetic extraction, and writes beside
# them: the released datasets cannot be overwritten by a demo run.
EXAMPLE = False

CONFIG = yaml.safe_load(open("config/config.yaml"))
TIME_WINDOW = int(CONFIG["time_window"])
LAST_YEAR = int(CONFIG["last_year"])
THRESHOLDS = CONFIG["preprocessing"]
RANKING = CONFIG["paths"]["raw_university_ranking"]

base = "data/example" if EXAMPLE else "data"
PANELS = {"timed": f"{base}/interim/panel.csv.gz", "snapshot": f"{base}/interim/panel_snapshot.csv.gz"}
OUT = {
    "window": f"{base}/processed/dataset_window.csv",
    "nowindow": f"{base}/processed/dataset_nowindow.csv",
}

pl.Config.set_tbl_cols(10)
print("panels     :", PANELS["timed"], "and", PANELS["snapshot"])
print("time window:", TIME_WINDOW, "years, up to", LAST_YEAR)
print("thresholds :", THRESHOLDS)

panels     : data/interim/panel.csv.gz and data/interim/panel_snapshot.csv.gz
time window: 7 years, up to 2024
thresholds : {'missing_threshold': 0.5, 'max_missing_per_row': 6}


---
## The two panels

They have to carry the same firm-years in the same order: the switches change
values, never rows. The two controls further down swap the target between the two
datasets on `CompanyID`, and that is only legitimate if this holds.

In [2]:
timed = pl.read_csv(PANELS["timed"], null_values=["NA"])
snapshot = pl.read_csv(PANELS["snapshot"], null_values=["NA"])
keys = ["CompanyID", "Age"]
print(f"timed   : {timed.height:,} rows x {timed.width} columns")
print(f"snapshot: {snapshot.height:,} rows x {snapshot.width} columns")
print("same firm-years:", timed.select(keys).equals(snapshot.select(keys)))

timed   : 802,148 rows x 52 columns
snapshot: 802,148 rows x 52 columns
same firm-years: True


The panel carries more than the models read: `TARGET_COLUMNS` are the columns that
place a firm in time and carry its outcome, `FEATURE_COLUMNS` are what the models
see. Everything else the panel holds is there for the construction, or declared as
an addition.

In [3]:
print(f"{len(FEATURE_COLUMNS)} features, {len(TARGET_COLUMNS)} columns for the target")
print("carried but not read:", sorted(set(timed.columns) - set(FEATURE_COLUMNS) - set(TARGET_COLUMNS)))

47 features, 6 columns for the target
carried but not read: ['N_Similar']


---
## The bias-controlled dataset

**The sample.** Only the firms that were in an early stage at least once, and that
reached it **within two years** of being founded: the prediction is made from the
first early-stage year, so a firm that only reaches it at age five would be
predicted from a very different point in its life.

**The decision point** is that first early-stage year, and every feature is read
there. **The target** is read at the decision point plus the window, or at the
last year observed if the panel is shorter: does the firm reach the next stage
within seven years? If the change happens later, at the decision point it is not
knowable, and the firm counts as not having moved.

**The calendar** matters too: only decision years from 2010 on, and only those that
leave a full window of future inside the extraction.

In [4]:
windowed = build_windowed_dataset(timed, TIME_WINDOW, LAST_YEAR)
print(f"firms: {windowed.height:,}")
print(windowed["Target"].value_counts(sort=True))
windowed.select("CompanyID", "Age", "StartingAge", "TargetAge", "GrowthStageGroup", "Target").head(5)

firms: 32,752
shape: (4, 2)
┌────────┬───────┐
│ Target ┆ count │
│ ---    ┆ ---   │
│ str    ┆ u32   │
╞════════╪═══════╡
│ Early  ┆ 13958 │
│ Out    ┆ 8659  │
│ Later  ┆ 6588  │
│ Exit   ┆ 3547  │
└────────┴───────┘


CompanyID,Age,StartingAge,TargetAge,GrowthStageGroup,Target
i64,i64,i64,i64,str,str
1,0,0,0,"""Early""","""Exit"""
2,0,0,2,"""Early""","""Out"""
3,0,0,1,"""Early""","""Out"""
4,1,1,8,"""Early""","""Later"""
5,0,0,7,"""Early""","""Early"""


**The preprocessing.** The institutes become one flag, "somebody from a top-50
university", because the raw column is a concatenation of names. The gender of the
chief executive becomes one indicator. The two categorical features are left as
raw categories on purpose: they are frequency-encoded **per split**, inside
`get_split`, so that a held-out row never contributes to its own encoding.

Then the missing-value policy. A feature missing on more than half the rows is
dropped: past that point the imputation would be inventing the column rather than
completing it. The per-row budget goes with it — with the threshold at 0.5 the four
sparsest features stay in, so a row missing exactly those is normal rather than
pathological. Both numbers live in `config.yaml`.

Finally the target becomes binary: reaching a later stage or an exit is a one,
staying or failing is a zero.

In [5]:
window = preprocess_dataset(
    windowed,
    RANKING,
    missing_threshold=THRESHOLDS["missing_threshold"],
    max_missing_per_row=THRESHOLDS["max_missing_per_row"],
)
dropped = sorted(set(FEATURE_COLUMNS) - set(window.columns))
print(f"dataset: {window.height:,} firms x {window.width} columns, base rate {window['Target'].mean():.3f}")
print("features dropped by the missing-value threshold:", dropped)

dataset: 30,007 firms x 49 columns, base rate 0.327
features dropped by the missing-value threshold: ['Gender_CEO', 'Institute']


---
## The dataset with the look-ahead

The same firms, described from the **snapshot** panel and over their whole observed
life: the amounts are summed, the shares and the indices averaged, and every other
column is taken at the last year on record. The target is whether the firm *ever*
reaches the next stage, with no horizon at all.

It is built against the windowed dataset and then reduced to its rows and columns,
because every comparison of the paper assumes the two carry the same firms
described in two ways.

In [6]:
full_history = build_full_history_dataset(snapshot, window)
nowindow = preprocess_dataset(
    full_history,
    RANKING,
    flag_no_time_window=True,
    missing_threshold=THRESHOLDS["missing_threshold"],
    max_missing_per_row=THRESHOLDS["max_missing_per_row"],
)
nowindow = nowindow.filter(pl.col("CompanyID").is_in(window["CompanyID"].implode())).select(window.columns)
print(f"dataset: {nowindow.height:,} firms x {nowindow.width} columns, base rate {nowindow['Target'].mean():.3f}")
print("same firms as the windowed one:", sorted(window["CompanyID"]) == sorted(nowindow["CompanyID"]))

dataset: 30,007 firms x 49 columns, base rate 0.410
same firms as the windowed one: True


The gap between the two base rates is not noise: it is the label leak. The target
of the second is easier to reach, because "ever" has no deadline. No model here
tunes its decision threshold, so F1, precision and recall move with the base rate
on their own, and **AUC is the metric to read across this axis**.

In [7]:
comparison = pl.DataFrame(
    {
        "dataset": ["window", "nowindow"],
        "firms": [window.height, nowindow.height],
        "columns": [window.width, nowindow.width],
        "base rate": [window["Target"].mean(), nowindow["Target"].mean()],
    }
)
comparison

dataset,firms,columns,base rate
str,i64,i64,f64
"""window""",30007,49,0.32729
"""nowindow""",30007,49,0.410338


---
## The same two datasets, in one call

Everything above is what `build_processed_datasets` does, and it is what the
command line calls: the steps are laid out here to be read, and composed there to
be run. The check below is the point — the two routes have to agree.

In [8]:
window_2, nowindow_2 = build_processed_datasets(
    timed,
    snapshot,
    RANKING,
    time_window=TIME_WINDOW,
    last_year=LAST_YEAR,
    missing_threshold=THRESHOLDS["missing_threshold"],
    max_missing_per_row=THRESHOLDS["max_missing_per_row"],
)
print("same windowed dataset :", window.equals(window_2))
print("same full-history one :", nowindow.equals(nowindow_2))

same windowed dataset : True
same full-history one : True


In [9]:
import pathlib

for name, dataset in (("window", window), ("nowindow", nowindow)):
    out = pathlib.Path(OUT[name])
    out.parent.mkdir(parents=True, exist_ok=True)
    dataset.write_csv(out)
    print("written:", out)

written: data/processed/dataset_window.csv


written: data/processed/dataset_nowindow.csv


---
## Next

**`3_experiments.ipynb`** reads these two files, assembles the six experiments out
of them — the two datasets, the two ablations and the two controls — trains the
seven model families and compares them.